# Padding
We wish to build the real function

$$f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x)=\sum_{k\in{\mathbb{Z}}}\,c[k]\,\varphi(x-k).$$
There, the sequence $c$ of coefficients is used to parameterize the function $f$ and gives us a good many degrees of freedom to shape it to our taste. Moreover, the basis $\varphi$ is assumed to have the desirable technical property of being a Riesz basis. Among other things, this ensures that the infinite sum found in the construction of $f$ is always well-behaved.

The adaptability of $c$ makes it a tool of choice to represent sampled data as the continuously defined function $f.$ Given a sequence $y$ of regularly indexed samples $y[q]$ for $q\in{\mathbb{Z}},$ the so-called interpolation condition leads to a procedure that determines $c$ such that $f(q)=y[q].$ Fortunately, the assumption of a Riesz basis guarantees the existence of $c$ for any $y.$ Furthermore, in case $\varphi$ is a B-spline, there exist very efficient algorithms to get $c$ out of $y.$

Unfortunately, the theoretical derivations of the algorithms rely on $y$ being a *sequence*, which means that infinitely many samples are required. Now, one never has access to a sequence of samples in practice, only to a finite-dimensional *vector* ${\mathbf{y}}\in{\mathbb{R}}^{K}$ of samples. To take advantage of the theoretical derivations of the efficient algorithms, it is thus an unavoidable necessity that a procedure be engineered that converts the vector ${\mathbf{y}}$ into the sequence $y.$ We call padding the operation that consists in the engineering of the subsequences $\left(y[k]\right)_{k\in{\mathbb{Z}}_{<0}}$ to the left and $\left(y[k]\right)_{k\in{\mathbb{Z}}_{\geq K}}$ to the right of the provided $\left(y[k]\right)_{k=0}^{K-1}$.

The engineering of ${\mathbf{y}}\mapsto y$ is application-dependent. For instance, ${\mathbf{y}}$ could represent angular data, in which case one would have to cope with angular wrapping. Or, ${\mathbf{y}}$ could represent intensity data, in which case one would have to discourage negative intensities in the continuously defined function $f$ being constructed through the steps ${\mathbf{y}}\mapsto y\mapsto c\mapsto f.$ Or, one could pretend that all unobserved samples do vanish and take the special value $0.$ Or, one could assume that the first observed sample has indeed the same value as all unobserved samples that came before, while the last observed sample has the same value as all unobserved samples that folllow.

The unobserved samples are, well, unobserved. Consequently, every strategy that assigns specific values to them is valid, but some are less practical than others. The ``splinekit`` library deals with paddings of low complexity; in particular, we focus on some for which the overall organization of $y,$ $c,$ and $f$ is consistent. The seven paddings being considered are

*   Periodic
*   Narrow Mirror
*   Wide Mirror
*   Anti-Mirror
*   Nega-Periodic
*   Nega-Narrow Mirror
*   Nega-Wide Mirror

Except for the anti-mirror padding, all forms are openly periodic. We illustrate now visually the effect of the various paddings on random splines of the specified degree. The data samples are represented with circles. The first sample, as well as its replicates when the padding is globally periodic, is indicated by a red stem line. The portion of curve in thick green is tied to the observed data, with a number of samples that can be specified. The portion of curve in thick blue is the complement to a full period.

In [ ]:
# Load the required libraries.
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_degree = 5 # Maximal spline degree
max_samples = 12 # Maximal support of the observed data samples
c0 = np.random.standard_normal(6) # Random coefficients
s0 = sk.PeriodicSpline1D.from_spline_coeff(c0, degree = 3)

# Plot
def update_plot (
    degree = 3,
    samples = 6,
    padding = 0
):
    global c0
    global s0
    if s0.degree != degree:
        s0 = s0.projected(degree = degree)
        c0 = s0.spline_coeff[ : len(c0)]
    if len(c0) < samples:
        c0 = np.append(c0, np.random.standard_normal(samples - len(c0)))
    else:
        c0 = c0[ : samples]
    highlight = sk.interval.Empty()
    downlight = sk.interval.Empty()
    s = sk.PeriodicSpline1D()
    plt_domain = sk.interval.Empty()
    if 0 == padding: # Periodic
        s = sk.PeriodicSpline1D.from_spline_coeff(c0, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c0) - 0.5, 2 * len(c0) + 0.5))
        highlight = sk.interval.ClosedOpen((0, len(c0)))
    elif 1 == padding: # Narrow Mirror
        c = np.zeros(2 * samples - 2, dtype = float)
        for k in range(len(c)):
            c[k] = sk.pad_n(c0, at = k)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c) - 0.5, 2 * len(c) + 0.5))
        highlight = sk.interval.ClosedOpen((0, len(c0) - 1))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(c)))
    elif 2 == padding: # Wide Mirror
        c = np.zeros(2 * samples, dtype = float)
        for k in range(len(c)):
            c[k] = sk.pad_w(c0, at = k)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c) - 0.5, 2 * len(c) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, len(c0) - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(c) - 0.5))
    elif 3 == padding: # Anti-Mirror
        c = np.zeros(degree + 1 + 3 * (2 * samples - 2) + degree + 1, dtype = float)
        delay = -degree - 1 - 2 * samples + 2
        for k in range(len(c)):
            c[k] = sk.pad_a(c0, at = k - delay)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree, delay = delay)
        plt_domain = sk.interval.ClosedOpen((
            -2 * samples + 2 - 0.5,
            4 * samples - 4 + 0.5
        ))
        highlight = sk.interval.ClosedOpen((0, len(c0) - 1))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, 2 * samples - 2))
    elif 4 == padding: # Nega-Periodic
        c = np.zeros(2 * samples, dtype = float)
        for k in range(len(c)):
            c[k] = sk.pad_np(c0, at = k)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c) - 0.5, 2 * len(c) + 0.5))
        highlight = sk.interval.ClosedOpen((0, len(c0)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(c)))
    elif 5 == padding: # Nega-Narrow Mirror
        c = np.zeros(2 * samples + 2, dtype = float)
        for k in range(len(c)):
            c[k] = sk.pad_nn(c0, at = k)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c) - 0.5, 2 * len(c) + 0.5))
        highlight = sk.interval.ClosedOpen((-1, len(c0)))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(c) - 1))
    elif 6 == padding: # Nega-Wide Mirror
        c = np.zeros(2 * samples, dtype = float)
        for k in range(len(c)):
            c[k] = sk.pad_nw(c0, at = k)
        s = sk.PeriodicSpline1D.from_spline_coeff(c, degree = degree)
        plt_domain = sk.interval.ClosedOpen((-len(c) - 0.5, 2 * len(c) + 0.5))
        highlight = sk.interval.ClosedOpen((-0.5, len(c0) - 0.5))
        downlight = sk.interval.ClosedOpen((highlight.rightbound, len(c) - 0.5))
    (fig, ax) = plt.subplots()
    s.plot((fig, ax), plotdomain = plt_domain, plotpoints = 301, knot_marker = "None")
    s.plot(
        (fig, ax),
        plotdomain = highlight,
        line_fmt = "-C2",
        line_width = 3,
        plotpoints = 101,
        knot_marker = "None"
    )
    if 0 < downlight.diameter:
        s.plot(
            (fig, ax),
            plotdomain = downlight,
            line_fmt = "-C0",
            line_width = 3,
            plotpoints = 101,
            knot_marker = "None"
        )
    plt.show()

widgets.interactive(
    update_plot,
    degree = (0, max_degree),
    samples = (1, max_samples),
    padding = widgets.RadioButtons(
        options = [
            ("Periodic", 0),
            ("Narrow Mirror", 1),
            ("Wide Mirror", 2),
            ("Anti-Mirror", 3),
            ("Nega-Periodic", 4),
            ("Nega-Narrow Mirror", 5),
            ("Nega-Wide Mirror", 6)
        ],
        value = 0,
        description = "Padding:",
        disabled = False
    )
)


## Periodic Padding
An easy, general-purpose padding approach is to engineer the sequence $y$ of samples as the straighforward periodized version of the vector ${\mathbf{y}}\in{\mathbb{R}}^{K}.$ This implies that the sequence $c$ of coefficients is $K$-periodic, too, for any basis $\varphi.$ Ultimately, the function $f$ is itself $K$-periodic. In summary, the relations being satisfied for any $k\in{\mathbb{Z}}$ and any $x\in{\mathbb{R}}$ are

$$\begin{eqnarray*}y[k]&=&y[k+K]\\c[k]&=&c[k+K]\\f(x)&=&f(x+K).\end{eqnarray*}$$

### Algorithmic Considerations
In the context of a periodic padding, there are three major algorithmic approaches to the solution of the interpolation constraint $f(q)=y[q]$ for $q\in[0\ldots K-1].$

*   Linear Algebra
*   Discrete Fourier
*   Recursive Filtering

The linear-algebra approach first establishes an explicit system of $K$ linear equations. The $q$-th equation of the system would be $y[q]=\sum_{k=0}^{K-1}\,\left(\sum_{p\in{\mathbb{Z}}}\,\varphi(q-p\,K-k)\right)\,c[k].$ Tools of linear algebra would then be deployed to solve the system in terms of the unknown variables $c[k].$ For general solvers, the overall computational cost is ${\mathcal{O}}(K^{3}).$

The discrete-Fourier approach is best described concisely in matrix notations. Let ${\mathbf{F}}\in{\mathbb{C}}^{K\times K}$ be the discrete Fourier transform, with the $\nu$-th row and $q$-th column entry given by ${\mathrm{e}}^{-{\mathrm{j}}\,\left(\nu-1\right)\,\frac{2\,\uppi}{K}\,\left(q-1\right)}$. Let the vector ${\mathbf{c}}$ represent one period of the periodic sequence $c.$ Moreover, let ${\mathbf{\upvarphi}}=(\sum_{p\in{\mathbb{Z}}}\,\varphi(p\,K+q))_{q=0}^{K-1}$ be the data-independent vector that contains the samples at the integers of the periodized basis $\varphi.$ Then, one has that ${\mathbf{c}}={\mathbf{F}}^{-1}\,\left(\left({\mathbf{F}}\,{\mathbf{y}}\right)\oslash\left({\mathbf{F}}\,{\mathbf{\upvarphi}}\right)\right),$ where $\oslash$ is an element-wise division. In practice, the Fourier transformation and its inverse are implemented via the fast Fourier algorithm, in which case the overall computational cost is ${\mathcal{O}}(K\,\log K).$

The recursive-filtering approach is the one followed in the ``splinekit`` library. It requires that the basis $\varphi$ has a finite support, is even-symmetric, and that the poles of the reciprocal of the $z$-transform of its samples at the integers are real numbers. These properties are all satisfied by the polynomial B-splines of nonnegative integer degree $n\in{\mathbb{N}}+2.$ Start the algorithm by letting ${\mathbf{c}}={\mathbf{y}}$. Then, iteratively for every one of the $\left\lfloor n/2\right\rfloor$ poles $z\in(-1,0),$ apply the in-place recursive updates

$$\left\{\begin{array}{rcll}c[0]&\leftarrow&\frac{1}{1-z^{K}}\,\left(c[0]+\sum_{k=1}^{K-1}\,z^{k}\,c[K-k]\right)\\c[k]&\leftarrow&c[k]+z\,c[k-1],&k\in[1\ldots K-1]\\c[K-1]&\leftarrow&\frac{\left(1-z\right)^{2}}{1-z^{K}}\,\left(c[K-1]+\sum_{k=0}^{K-2}\,z^{k+1}\,c[k]\right)\\c[K-1-k]&\leftarrow&z\,c[K-k]+\left(1-z\right)^{2}\,c[K-1-k],&k\in[1\ldots K-1].\end{array}\right.$$
The overall computational cost is now ${\mathcal{O}}(K\,\left\lfloor n/2\right\rfloor).$ In practice, further acceleration can be achieved if the sums that appear in the recursive-update equations are truncated at that index $k$ where the term $z^{k}$ becomes negligible.